In [198]:
import pickle

results_gpt_5 = pickle.load(open("traces_gpt-5.pkl",'rb'))
results_deepseek_r1 = pickle.load(open("traces_deepseek-r1-0528.pkl",'rb'))
results_ds_distill_qwen = pickle.load(open("traces_deepseek-ai_DeepSeek-R1-Distill-Qwen-32B.pkl",'rb'))
results_llama_70 = pickle.load(open("traces_meta-llama_Meta-Llama-3.1-70B-Instruct.pkl",'rb'))
results_llama_8 = pickle.load(open("traces_meta-llama_Meta-Llama-3.1-8B-Instruct.pkl",'rb'))
results_qwen_06 = pickle.load(open("traces_Qwen_Qwen3-0.6B.pkl",'rb'))
results_qwen_4 = pickle.load(open("traces_Qwen_Qwen3-4B.pkl",'rb'))
results_qwen_8 = pickle.load(open("traces_Qwen_Qwen3-8B.pkl",'rb'))

In [ ]:
def get_answer_qwen(sample_text, include_thinking = False):
    if not include_thinking:
        if "</think>" in sample_text:
            answer_start = sample_text.index("</think>") + len("</think>\n")
        elif "<think>" not in sample_text:
            answer_start = sample_text.index("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        else:
            return None
    else:
        answer_start = sample_text.index("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    if "<|im_end|>" not in sample_text[answer_start:]:
        return sample_text[answer_start:]
    else:
        answer_end = sample_text.index("<|im_end|>", answer_start)
        return sample_text[answer_start:answer_end]

def get_answer_deepseek(sample_text, include_thinking = False):
    if not include_thinking:
        answer_start = 0
        if "<think>" not in sample_text:
            answer_start = sample_text.index("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        else:
            if "</think>" not in sample_text:
                return None
            while "</think>" in sample_text[answer_start:]:
                answer_start = sample_text.index("</think>", answer_start) + len("</think>\n")
    else:
        answer_start = sample_text.index("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    
    if "<｜end▁of▁sentence｜>" not in sample_text[answer_start:]:
        return sample_text[answer_start:]
    else:
        answer_end = sample_text.index("<｜end▁of▁sentence｜>", answer_start)
        return sample_text[answer_start:answer_end]
    
def get_answer_llama(sample_text):
    ass_start_tok = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    answer_start = sample_text.index(ass_start_tok) + len(ass_start_tok)
    if "<|eot_id|>" not in sample_text[answer_start:]:
        return sample_text[answer_start:]
    else:
        answer_end = sample_text.index("<|eot_id|>", answer_start)
        return sample_text[answer_start:answer_end]

def answer_function(model_name):
    model_name = model_name.lower()
    if "deepseek" in model_name:
        return lambda x: get_answer_deepseek(x, include_thinking=True)
    elif "qwen" in model_name:
        return get_answer_qwen
    elif "llama" in model_name:
        return get_answer_llama
    elif "gpt-5" in model_name:
        return lambda x, _: x
    else:
        print("Warning: Unknown model name. Defaulting to identity function.")
        return lambda x, _: x


In [287]:
def produce_diffed_data(file1, file2):
    joint_data = []
    judge_wrong = []
    judge_right = []
    both_right = []
    both_wrong = []
    model_1, model_2 = file1['metadata']['model_name'], file2['metadata']['model_name']
    data_1, data_2 = file1['data'], file2['data']
    completes = 0
    for i in range(len(data_1['questions'])):
        assert data_1['questions'][i] == data_2['questions'][i], f"Questions do not match at index {i}"
        assert data_1['ground_truth_answers'][i] == data_2['ground_truth_answers'][i], f"Answers do not match at index {i}"
        completions_1, completions_2 = data_1['completions'][i], data_2['completions'][i]
        completions_1_clean = answer_function(model_1)(completions_1)
        completions_2_clean = answer_function(model_2)(completions_2)
        if completions_1_clean is None or completions_2_clean is None:
            continue
        completes += 1
        ea_1, ea_2 = data_1['extracted_answers'][i], data_2['extracted_answers'][i]
        s_1, s_2 = data_1['scores'][i], data_2['scores'][i]
        if s_1 != s_2:
            joint_data.append({
                'question': data_1['questions'][i],
                'ground_truth_answer': data_1['ground_truth_answers'][i],
                'judge_completion': completions_1_clean,
                'judge_extracted_answer': ea_1,
                'judge_score': s_1,
                'ref_completion': completions_2_clean,
                'ref_extracted_answer': ea_2,
                'ref_score': s_2,
            })
            if s_1 == 1 and s_2 == 0:
                judge_right.append(i)
            elif s_1 == 0 and s_2 == 1:
                judge_wrong.append(i)
        elif s_1 == 1 and s_2 == 1:
            both_right.append(i)
        elif s_1 == 0 and s_2 == 0:
            both_wrong.append(i)
        else:
            raise ValueError(f"Unexpected scores at index {i}: {s_1}, {s_2}")
    payload = {
        'metadata': {
            'judge': model_1,
            'ref': model_2,
            'total_examples': completes,
            'differing_examples': len(joint_data),
            'num_right': len(judge_right),
            'num_wrong': len(judge_wrong),
            'num_both_right': len(both_right),
            'num_both_wrong': len(both_wrong),
        },
        'legit_idx': judge_right,
        'illegit_idx': judge_wrong,
        'both_right_idx': both_right,
        'both_wrong_idx': both_wrong,
        'data': joint_data
    }
    return payload

In [288]:
diff_data = produce_diffed_data(results_llama_70, results_ds_distill_qwen)

In [289]:
diff_data['data'][diff_data['illegit_idx'][0]]

{'question': 'Four distinct integers $a$, $b$, $c$ and $d$ have the property that when added in pairs, the sums 10, 18, 19, 20, 21, and 29 are obtained. What are the four integers in increasing order? (place a comma and then a space between each integer)',
 'ground_truth_answer': '4,6,14,15',
 'judge_completion': "Since there are six possible pairs, we can use a system to organize these six equations and narrow down the values of $a$, $b$, $c$, and $d$. \n\n$$a+b=10$$\n$$a+c=19$$\n$$a+d=21$$\n$$b+c=18$$\n$$b+d=29$$\n$$c+d=20$$\n\nTaking the first three equations, we have:\n$a=10-b$\n$a=19-c$\n$a=21-d$\n\nFrom $a=10-b$ and $a=19-c$,\n$$10-b=19-c$$\n$$c=9+b$$\n\nFrom $a=10-b$ and $a=21-d$,\n$$10-b=21-d$$\n$$d=b+11$$\n\nSubstitute $c=9+b$ and $d=b+11$ into the remaining three equations:\n\n$$b+9+b=18$$\n$$b+b+11=29$$\n$$9+b+b+11=20$$\n\nSolving for $b$, we find there is only one valid solution for $b$.\n$$2b=9$$\n$$b=\\frac{9}{2}$$ is not valid since the problem specifies integers.\n\nHow

In [ ]:
diff_data

In [97]:
ask = [k for k in results_2['data']['completions']  if "</think>" not in k and "<think>" in k]

In [115]:
len(ask[10])

6111